# Hafta 5 — Sınıflandırma ve Regresyon Metrikleri

## Bir modelin iyi olup olmadığına nasıl karar veririm?

Bu hafta vize öncesindeki son derste model kurmaktan çok, kurduğum modelin **hangi açıdan iyi veya kötü olduğunu** anlamaya odaklanıyorum.

Bir metriği sadece yüksek veya düşük olduğu için yorumlamıyorum. Şu üç soruyu birlikte soruyorum:

1. Model hangi hatayı yapıyor?
2. Bu hatanın iş maliyeti nedir?
3. Seçtiğim metrik bu maliyeti gerçekten görünür kılıyor mu?

Notebook tamamen kendi içinde çalışır; harici veri dosyasına ihtiyaç duymaz.


## Dersin hedefleri

Bu dersin sonunda:

- sınıflandırmada confusion matrix'i okuyabileceğim,
- accuracy, precision, recall, specificity, F1, balanced accuracy ve MCC arasındaki farkı açıklayabileceğim,
- ROC-AUC ile PR-AUC'nin aynı şeyi ölçmediğini gösterebileceğim,
- log-loss ve Brier score ile olasılık tahminlerini değerlendirebileceğim,
- regresyonda MAE, MSE, RMSE, $R^2$, MAPE, sMAPE, RMSLE ve WAPE'yi hesaplayabileceğim,
- metriği teknik alışkanlığa göre değil, iş problemine göre seçebileceğim.

### Önerilen ders akışı

- **İlk 15 dakika:** metrik, hata ve iş maliyeti
- **Sonraki 45 dakika:** sınıflandırma metrikleri
- **Sonraki 40 dakika:** regresyon metrikleri
- **Son 20 dakika:** metrik seçimi ve mini vaka


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from math import sqrt

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42


## 1. Metrik neden iş probleminden bağımsız değildir?

Aynı model çıktısı iki projede tamamen farklı kararlar doğurabilir.

- Dolandırıcılık projesinde bir olayı kaçırmak yüksek finansal kayıp yaratabilir. Burada **recall** kritik olabilir.
- Pazarlama kampanyasında yanlış kişiye teklif göndermek maliyet yaratabilir. Burada **precision** daha önemli olabilir.
- Hasar tutarı tahmininde birkaç çok büyük hata bütçeyi bozuyorsa **RMSE**, tipik hatayı anlatmak istiyorsam **MAE** daha anlamlı olabilir.

Bu nedenle önce metriği seçip sonra probleme bakmıyorum. Önce hata türünü ve maliyeti tanımlıyorum; metriği bunun ardından seçiyorum.


# Bölüm A — Sınıflandırma Metrikleri

Sınıflandırma modelim çoğu zaman önce bir olasılık üretir:

$$p_i = P(y_i=1\mid x_i)$$

Bu olasılığı bir karar eşiğiyle sınıfa çeviririm:

$$\hat{y}_i = \begin{cases}1, & p_i \ge t \\ 0, & p_i < t\end{cases}$$

Buradaki $t$ karar eşiğidir. Varsayılan olarak sıkça $0.50$ kullanılır; fakat bu sayı iş maliyetinden bağımsız bir doğa kanunu değildir.


In [ ]:
# Ders boyunca kullanacağım küçük sınıflandırma örneği
y_true = np.array([1, 0, 1, 0, 1, 0, 0, 1, 0, 1,
                   0, 0, 1, 0, 1, 0, 0, 1, 0, 1])

y_prob = np.array([0.92, 0.18, 0.81, 0.44, 0.73, 0.35, 0.12, 0.66, 0.28, 0.58,
                   0.62, 0.09, 0.49, 0.31, 0.88, 0.52, 0.21, 0.77, 0.41, 0.55])

threshold = 0.50
y_pred = (y_prob >= threshold).astype(int)

pd.DataFrame({"gercek": y_true, "olasilik": y_prob, "tahmin": y_pred}).head(10)


## 2. Confusion Matrix — bütün temel metriklerin başlangıç noktası

Confusion matrix bir skor değil, dört hata/doğru türünü birlikte gösteren tablodur.

| Gerçek / Tahmin | Pozitif | Negatif |
|---|---:|---:|
| Pozitif | TP — doğru pozitif | FN — kaçırılan pozitif |
| Negatif | FP — yanlış alarm | TN — doğru negatif |

- **TP:** Pozitif olayı doğru buldum.
- **FP:** Pozitif dedim ama olay yoktu.
- **FN:** Negatif dedim ama olay vardı.
- **TN:** Negatif olayı doğru bildim.

### İki proje çıktısını yorumlama

**Proje 1 — Hasar tespiti:** TP=80, FN=20 ise gerçekleşen 100 hasarın 20'sini kaçırmışım. Operasyon açısından FN'nin maliyetini ayrıca tartışmam gerekir.

**Proje 2 — Kampanya hedefleme:** TP=300, FP=700 ise 1.000 teklifin yalnızca 300'ü doğru kişiye gitmiş. Model fırsat yakalıyor olabilir; fakat teklif maliyeti yüksekse FP sayısı kabul edilemez olabilir.


In [ ]:
def confusion_counts(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    return tp, tn, fp, fn

tp, tn, fp, fn = confusion_counts(y_true, y_pred)
cm = np.array([[tn, fp], [fn, tp]])

fig, ax = plt.subplots(figsize=(5.5, 4.2))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Tahmin 0", "Tahmin 1"],
            yticklabels=["Gerçek 0", "Gerçek 1"], ax=ax)
ax.set_title(f"Confusion Matrix — eşik = {threshold:.2f}")
ax.set_xlabel("Model tahmini")
ax.set_ylabel("Gerçek sınıf")
plt.tight_layout()
plt.show()

print({"TP": tp, "TN": tn, "FP": fp, "FN": fn})


## 3. Accuracy — bütün tahminlerin ne kadarı doğru?

$$\text{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN}$$

### Adım adım

1. Doğru pozitifleri ve doğru negatifleri toplarım.
2. Toplam gözlem sayısına bölerim.
3. Sonucu tüm kararların doğruluk oranı olarak okurum.

Accuracy sınıflar dengeliyse ve FP ile FN'nin maliyeti benzerse kullanışlıdır. Dengesiz veride tek başına tehlikelidir.

### İki proje çıktısını yorumlama

**Proje 1 — Motor sigortası:** 10.000 poliçenin yalnızca 500'ünde hasar var. Herkese “hasar yok” diyen modelin accuracy'si $9.500/10.000=\%95$ olur. Buna rağmen model hiçbir hasarı bulamaz; bu sonuç başarılı değildir.

**Proje 2 — Belge sınıflandırma:** İki sınıf yaklaşık dengeli ve 2.000 belgenin 1.840'ı doğru sınıflanmışsa accuracy $1.840/2.000=\%92$'dir. Sınıf bazlı sonuçlar da yakınsa bu güçlü bir genel performans işaretidir.


In [ ]:
accuracy = (tp + tn) / (tp + tn + fp + fn)
print(f"Accuracy: {accuracy:.3f}")

# Dengesiz veri uyarısı
claim_actual = np.array([1] * 50 + [0] * 950)
claim_pred_all_zero = np.zeros(1000, dtype=int)
print("Herkese hasar yok diyen modelin accuracy değeri:",
      np.mean(claim_actual == claim_pred_all_zero))


## 4. Precision — pozitif dediklerimin ne kadarı gerçekten pozitif?

$$\text{Precision}=\frac{TP}{TP+FP}$$

Precision, modelin verdiği pozitif kararların **isabetini** ölçer. Yanlış alarmın maliyeti yüksek olduğunda önem kazanır.

### Adım adım

1. Modelin bütün pozitif tahminlerini $TP+FP$ ile sayarım.
2. Bunların içindeki doğru pozitifleri seçerim.
3. Doğru pozitifleri bütün pozitif tahminlere bölerim.

### İki proje çıktısını yorumlama

**Proje 1 — Dolandırıcılık inceleme kuyruğu:** TP=180 ve FP=60 ise precision $180/(180+60)=\%75$'tir. İncelemeye gönderilen her 4 işlemin 3'ü gerçekten şüphelidir; ekip kapasitesi verimli kullanılıyor olabilir.

**Proje 2 — Churn kampanyası:** TP=90 ve FP=210 ise precision $90/(90+210)=\%30$'dur. Teklif gönderdiğim 10 müşterinin yalnızca 3'ü gerçekten ayrılacaktır. Teklif pahalıysa eşik veya model yeniden değerlendirilmelidir.


In [ ]:
precision = tp / (tp + fp) if (tp + fp) else 0.0
print(f"Precision: {precision:.3f}")


## 5. Recall — gerçek pozitiflerin ne kadarını buldum?

Recall; sensitivity veya true positive rate adıyla da karşıma çıkar.

$$\text{Recall}=\frac{TP}{TP+FN}$$

Pozitif olayı kaçırmanın maliyeti yüksek olduğunda recall önemlidir.

### Adım adım

1. Gerçekte pozitif olan bütün gözlemleri $TP+FN$ ile sayarım.
2. Bunların kaçını doğru bulduğumu $TP$ ile sayarım.
3. $TP$'yi bütün gerçek pozitiflere bölerim.

### İki proje çıktısını yorumlama

**Proje 1 — Şebeke arızası erken uyarısı:** 120 gerçek arızanın 108'i bulunmuşsa recall $108/120=\%90$'dır. Model arızaların %10'unu kaçırıyor; kaçırılan 12 olayın operasyonel etkisi ayrıca hesaplanmalıdır.

**Proje 2 — Kalite kontrol:** 400 hatalı ürünün 240'ı tespit edilmişse recall $240/400=\%60$'tır. Hatalı ürünlerin %40'ı üretim hattından kaçmaktadır; precision yüksek olsa bile süreç için yetersiz olabilir.


In [ ]:
recall = tp / (tp + fn) if (tp + fn) else 0.0
print(f"Recall: {recall:.3f}")


## 6. Specificity — gerçek negatiflerin ne kadarını doğru eledim?

Specificity, true negative rate olarak da adlandırılır.

$$\text{Specificity}=\frac{TN}{TN+FP}$$

Negatif sınıfı yanlışlıkla pozitif işaretlemek istemediğimde specificity önemlidir.

### İki proje çıktısını yorumlama

**Proje 1 — Spam filtresi:** 9.000 gerçek normal e-postanın 8.820'si doğru bırakılmışsa specificity $8.820/9.000=\%98$'dir. Normal e-postaların %2'si yanlışlıkla spam'e gitmektedir.

**Proje 2 — Kredi risk ön elemesi:** 6.000 düzenli ödeme yapacak müşterinin 4.800'ü doğru biçimde düşük riskli bulunmuşsa specificity $4.800/6.000=\%80$'dir. İyi müşterilerin %20'si yanlış alarm nedeniyle elenebilir; bunun gelir kaybı hesaplanmalıdır.


In [ ]:
specificity = tn / (tn + fp) if (tn + fp) else 0.0
print(f"Specificity: {specificity:.3f}")


## 7. F1 Score — precision ve recall'un harmonik ortalaması

$$F_1=2\times\frac{\text{Precision}\times\text{Recall}}
{\text{Precision}+\text{Recall}}$$

Eşdeğer gösterim:

$$F_1=\frac{2TP}{2TP+FP+FN}$$

Harmonik ortalama düşük olan bileşeni daha fazla cezalandırır. Bu nedenle precision çok yüksek, recall çok düşük olduğunda F1 yüksek görünmez. F1, TN'yi doğrudan kullanmadığı için her problemde tek başına yeterli değildir.

### İki proje çıktısını yorumlama

**Proje 1 — Churn:** Precision=0,60 ve recall=0,75 ise $F_1\approx0,667$ olur. Model hem teklif isabeti hem de churn yakalama açısından orta-iyi bir denge kurmaktadır.

**Proje 2 — Dolandırıcılık:** Precision=0,90 ve recall=0,20 ise $F_1\approx0,327$ olur. Uyarılar çok isabetli olsa da vakaların %80'i kaçtığı için denge zayıftır.


In [ ]:
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
print(f"F1: {f1:.3f}")


## 8. Balanced Accuracy — iki sınıfa eşit söz hakkı vermek

$$\text{Balanced Accuracy}=\frac{\text{Recall}+\text{Specificity}}{2}$$

Normal accuracy çoğunluk sınıfından etkilenebilir. Balanced accuracy, pozitif ve negatif sınıftaki başarıyı eşit ağırlıkla ortalar.

### İki proje çıktısını yorumlama

**Proje 1 — Hasar tahmini:** Recall=0,40 ve specificity=0,96 ise balanced accuracy $(0,40+0,96)/2=0,68$'dir. Normal accuracy çok yüksek olsa bile pozitif sınıf performansının zayıflığı görünür olur.

**Proje 2 — Dengeli belge sınıflandırma:** Recall=0,91 ve specificity=0,89 ise balanced accuracy 0,90'dır. Her iki sınıfta da benzer ve güçlü performans vardır.


In [ ]:
balanced_accuracy = (recall + specificity) / 2
print(f"Balanced accuracy: {balanced_accuracy:.3f}")


## 9. MCC — dört hücreyi birlikte kullanan korelasyon benzeri skor

Matthews Correlation Coefficient:

$$MCC=\frac{TP\times TN-FP\times FN}
{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

- $+1$: kusursuz tahmin
- $0$: rastgele ilişkiye yakın
- $-1$: tamamen ters tahmin

MCC, sınıf dengesizliğinde bütün confusion matrix hücrelerini kullandığı için faydalıdır. Yine de skorun iş maliyetini tek başına anlatmadığını unutmam.

### İki proje çıktısını yorumlama

**Proje 1 — Dolandırıcılık:** MCC=0,61 ise modelin tahminleri ile gerçek sınıflar arasında güçlü sayılabilecek pozitif ilişki vardır. Eşikler iş maliyetiyle ayrıca kontrol edilmelidir.

**Proje 2 — Hasar oluşumu:** Accuracy=0,94 fakat MCC=0,08 ise yüksek accuracy büyük ölçüde çoğunluk sınıfından gelmektedir; model ayrıştırma açısından zayıftır.


In [ ]:
den = sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
mcc = (tp * tn - fp * fn) / den if den else 0.0
print(f"MCC: {mcc:.3f}")


## 10. ROC-AUC — model pozitifleri negatiflerin önüne koyabiliyor mu?

ROC eğrisi farklı eşiklerde şu iki değeri karşılaştırır:

$$TPR=\text{Recall}=\frac{TP}{TP+FN}$$

$$FPR=\frac{FP}{FP+TN}=1-\text{Specificity}$$

ROC-AUC, ROC eğrisinin altında kalan alandır. Başka bir güçlü yorum şudur: rastgele seçilen bir pozitif ve bir negatif gözlem için modelin pozitife daha yüksek skor verme olasılığıdır.

- 0,50: rastgele sıralamaya yakın
- 1,00: kusursuz sıralama

Bu aralıkları evrensel kalite etiketi olarak kullanmıyorum; veri, dönem ve iş maliyeti önemlidir.

### İki proje çıktısını yorumlama

**Proje 1 — Churn:** ROC-AUC=0,82 ise rastgele seçilen churn müşterisinin skoru, rastgele seçilen kalacak müşteriden yaklaşık %82 olasılıkla daha yüksektir. Bu değer doğrudan “%82 doğru tahmin” demek değildir.

**Proje 2 — Hasar:** ROC-AUC=0,76 fakat pozitif oranı %3 ise model sıralama yapabiliyor olabilir. Operasyonel eşikte precision ve recall'u ayrıca incelemem gerekir; ROC-AUC tek başına alarm kalitesini göstermez.


In [ ]:
def roc_points(y_true, y_score):
    thresholds = np.r_[np.inf, np.sort(np.unique(y_score))[::-1], -np.inf]
    rows = []
    for t in thresholds:
        pred = (y_score >= t).astype(int)
        tp_, tn_, fp_, fn_ = confusion_counts(y_true, pred)
        tpr = tp_ / (tp_ + fn_) if (tp_ + fn_) else 0.0
        fpr = fp_ / (fp_ + tn_) if (fp_ + tn_) else 0.0
        rows.append((t, fpr, tpr))
    return pd.DataFrame(rows, columns=["threshold", "fpr", "tpr"]).drop_duplicates(["fpr", "tpr"])

roc_df = roc_points(y_true, y_prob).sort_values(["fpr", "tpr"])
roc_auc = np.trapezoid(roc_df["tpr"], roc_df["fpr"])

plt.figure(figsize=(6, 5))
plt.plot(roc_df["fpr"], roc_df["tpr"], marker="o", label=f"ROC-AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Rastgele sıralama")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate / Recall")
plt.title("ROC Eğrisi")
plt.legend()
plt.tight_layout()
plt.show()


## 11. PR-AUC — pozitif sınıf az olduğunda alarm kalitesi

Precision–Recall eğrisi farklı eşiklerde precision ve recall ilişkisini gösterir.

Basamaklı alan yaklaşımında:

$$AP=\sum_k (R_k-R_{k-1})P_k$$

Burada $P_k$ precision, $R_k$ recall değeridir. PR-AUC/AP için önemli referans, pozitif sınıf oranıdır. Pozitif oranı %5 ise rastgele modelin precision seviyesi yaklaşık %5'tir.

### İki proje çıktısını yorumlama

**Proje 1 — Dolandırıcılık:** Pozitif oranı %1, PR-AUC=0,28 ise skor 0,28 “düşük” görünse de rastgele başlangıç 0,01'dir. Model nadir vakaları sıralamada anlamlı sinyal taşıyor olabilir.

**Proje 2 — Ürün önerisi:** Pozitif oranı %30, PR-AUC=0,38 ise rastgele başlangıç zaten 0,30'dur. 0,38'in operasyonel değeri sınırlı olabilir; lift ve maliyet analizi gerekir.


In [ ]:
def pr_points(y_true, y_score):
    thresholds = np.sort(np.unique(y_score))[::-1]
    rows = [(np.inf, 1.0, 0.0)]
    for t in thresholds:
        pred = (y_score >= t).astype(int)
        tp_, tn_, fp_, fn_ = confusion_counts(y_true, pred)
        p = tp_ / (tp_ + fp_) if (tp_ + fp_) else 1.0
        r = tp_ / (tp_ + fn_) if (tp_ + fn_) else 0.0
        rows.append((t, p, r))
    return pd.DataFrame(rows, columns=["threshold", "precision", "recall"])

pr_df = pr_points(y_true, y_prob).sort_values("recall")
delta_recall = pr_df["recall"].diff().fillna(0).clip(lower=0)
average_precision = float(np.sum(delta_recall * pr_df["precision"]))
baseline = float(np.mean(y_true))

plt.figure(figsize=(6, 5))
plt.step(pr_df["recall"], pr_df["precision"], where="post",
         label=f"AP ≈ {average_precision:.3f}")
plt.axhline(baseline, linestyle="--", color="gray",
            label=f"Pozitif oranı = {baseline:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Eğrisi")
plt.legend()
plt.tight_layout()
plt.show()


## 12. Log-Loss — doğru sınıf kadar güven seviyesi de önemli

Binary log-loss:

$$\text{Log-Loss}=-\frac{1}{n}\sum_{i=1}^{n}
\left[y_i\log(p_i)+(1-y_i)\log(1-p_i)\right]$$

### Adım adım

1. Gerçek sınıf 1 ise $-\log(p_i)$ cezasını kullanırım.
2. Gerçek sınıf 0 ise $-\log(1-p_i)$ cezasını kullanırım.
3. Cezaları ortalarım.

Yanlış ve aşırı güvenli tahminler çok büyük ceza alır. Log-loss küçüldükçe daha iyidir; 0 kusursuz olasılıklara yaklaşır.

### İki proje çıktısını yorumlama

**Proje 1 — Churn olasılığı:** Model A ve B aynı accuracy'ye sahip olsun. A yanlış müşteriye 0,99 churn olasılığı, B ise 0,55 veriyorsa A daha ağır cezalandırılır. Çünkü A yanlış kararından aşırı emindir.

**Proje 2 — Hasar olasılığı:** Eski model log-loss=0,184, yeni model=0,139 ise yeni model gerçek sınıflara ortalama olarak daha uygun olasılıklar üretmektedir. Farkın zaman ayrımlı testte ve kalibrasyon grafiğinde doğrulanması gerekir.


In [ ]:
def binary_log_loss(y_true, y_prob, eps=1e-15):
    p = np.clip(np.asarray(y_prob), eps, 1 - eps)
    y = np.asarray(y_true)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

print(f"Log-loss: {binary_log_loss(y_true, y_prob):.4f}")

# Aynı doğru/yanlış karar, farklı güven
print("Yanlış ama %99 güvenli tahminin cezası:", -np.log(1 - 0.99))
print("Yanlış ama %55 güvenli tahminin cezası:", -np.log(1 - 0.55))


## 13. Brier Score — olasılık hatasının karesi

$$\text{Brier}=\frac{1}{n}\sum_{i=1}^{n}(p_i-y_i)^2$$

Brier score, tahmin olasılığı ile gerçekleşen 0/1 sonuç arasındaki karesel farkın ortalamasıdır. Küçük değer daha iyidir. Hem ayrıştırma hem kalibrasyondan etkilenir; sınıf oranı ve referans modelle birlikte yorumlanmalıdır.

### İki proje çıktısını yorumlama

**Proje 1 — Hasar olasılığı:** Basit sınıf-oranı modeli Brier=0,047, geliştirdiğim model Brier=0,032 üretiyorsa model olasılık hatasını azaltmıştır. Yine de güvenilirlik eğrisiyle hangi olasılık bantlarında sapma olduğunu kontrol ederim.

**Proje 2 — Churn:** Model A Brier=0,118, Model B Brier=0,104 ise B'nin olasılıkları ortalama karesel hata açısından daha iyidir. Fakat kampanya kararında eşik sonrası precision/recall da gereklidir.


In [ ]:
def brier_score(y_true, y_prob):
    return float(np.mean((np.asarray(y_prob) - np.asarray(y_true)) ** 2))

print(f"Brier score: {brier_score(y_true, y_prob):.4f}")


## 14. Çok sınıflı problemlerde macro, micro ve weighted ortalama

Birden fazla sınıf olduğunda her sınıfı sırayla “pozitif”, diğerlerini “negatif” kabul ederek sınıf bazlı precision, recall ve F1 hesaplayabilirim. Sonra bunları nasıl ortalayacağıma karar veririm.

### Macro ortalama

Her sınıfa eşit ağırlık verir:

$$F1_{macro}=\frac{1}{K}\sum_{k=1}^{K}F1_k$$

### Weighted ortalama

Her sınıfı gözlem sayısıyla ağırlıklandırır:

$$F1_{weighted}=\sum_{k=1}^{K}\frac{n_k}{n}F1_k$$

### Micro ortalama

Önce bütün sınıfların TP, FP ve FN değerlerini toplar, sonra metriği hesaplar:

$$F1_{micro}=\frac{2\sum_kTP_k}{2\sum_kTP_k+\sum_kFP_k+\sum_kFN_k}$$

Tek etiketli çok sınıflı problemde micro F1 çoğu zaman accuracy ile aynı değere gelir.

### İki proje çıktısını yorumlama

**Proje 1 — 15 çağrı nedeni:** Weighted F1=0,89 fakat macro F1=0,61 ise yüksek hacimli çağrı nedenleri iyi, az görülen nedenler zayıf sınıflanıyor olabilir. Sadece weighted F1 raporlamak nadir sınıfları gizler.

**Proje 2 — Beş dengeli belge sınıfı:** Micro F1=0,92, macro F1=0,91 ve weighted F1=0,92 ise performans sınıflar arasında büyük ölçüde tutarlıdır. Yine de sınıf bazlı confusion matrix'i kontrol ederim.


In [ ]:
# Sınıf bazlı F1 değerlerinden macro ve weighted ortalama
class_report = pd.DataFrame({
    "sinif": ["Fatura", "Teknik", "İptal", "Kampanya", "Diğer"],
    "support": [4_000, 3_000, 1_500, 1_000, 500],
    "f1": [0.94, 0.91, 0.73, 0.58, 0.31],
})

macro_f1 = class_report["f1"].mean()
weighted_f1 = np.average(class_report["f1"], weights=class_report["support"])

display(class_report)
print(f"Macro F1: {macro_f1:.3f}")
print(f"Weighted F1: {weighted_f1:.3f}")
print("Fark, yüksek hacimli sınıfların genel skoru yukarı taşıdığını gösteriyor.")


## 15. Karar eşiği değişince ne olur?

Eşiği düşürürsem daha fazla gözleme pozitif derim:

- TP genellikle artar,
- FP genellikle artar,
- recall genellikle artar,
- precision düşebilir.

Eşiği yükseltirsem bunun tersi olur. Eşik seçimini yalnızca F1'i maksimum yaparak değil, iş maliyetiyle değerlendirmek daha doğrudur.

Basit maliyet örneği:

$$\text{Toplam maliyet}=FP\times C_{FP}+FN\times C_{FN}$$


In [ ]:
def metrics_at_threshold(y_true, y_prob, threshold, fp_cost=1, fn_cost=5):
    pred = (y_prob >= threshold).astype(int)
    tp_, tn_, fp_, fn_ = confusion_counts(y_true, pred)
    precision_ = tp_ / (tp_ + fp_) if (tp_ + fp_) else 0.0
    recall_ = tp_ / (tp_ + fn_) if (tp_ + fn_) else 0.0
    f1_ = 2 * precision_ * recall_ / (precision_ + recall_) if (precision_ + recall_) else 0.0
    return {
        "threshold": threshold, "TP": tp_, "FP": fp_, "FN": fn_, "TN": tn_,
        "precision": precision_, "recall": recall_, "f1": f1_,
        "business_cost": fp_ * fp_cost + fn_ * fn_cost,
    }

threshold_table = pd.DataFrame([
    metrics_at_threshold(y_true, y_prob, t, fp_cost=1, fn_cost=5)
    for t in np.arange(0.20, 0.81, 0.05)
])

display(threshold_table.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(threshold_table["threshold"], threshold_table["precision"], marker="o", label="Precision")
axes[0].plot(threshold_table["threshold"], threshold_table["recall"], marker="o", label="Recall")
axes[0].plot(threshold_table["threshold"], threshold_table["f1"], marker="o", label="F1")
axes[0].set_title("Eşik değiştikçe metrikler")
axes[0].set_xlabel("Karar eşiği")
axes[0].legend()

axes[1].plot(threshold_table["threshold"], threshold_table["business_cost"], marker="o", color="firebrick")
axes[1].set_title("Örnek iş maliyeti")
axes[1].set_xlabel("Karar eşiği")
axes[1].set_ylabel("FP×1 + FN×5")
plt.tight_layout()
plt.show()


## Sınıflandırma için kısa seçim rehberi

| İş sorusu | Öncelikli metrikler | Neden? |
|---|---|---|
| Pozitif olayı kaçırmak pahalı mı? | Recall, PR-AUC | FN görünür olur |
| Yanlış alarm pahalı mı? | Precision, specificity | FP görünür olur |
| İki hata türü benzer mi? | F1, MCC | Dengeyi özetler |
| Sınıflar dengesiz mi? | PR-AUC, balanced accuracy, MCC | Çoğunluk sınıfı etkisini azaltır |
| Sıralama kalitesi mi önemli? | ROC-AUC, PR-AUC | Eşikler boyunca karşılaştırır |
| Olasılık kalitesi mi önemli? | Log-loss, Brier score | Güven seviyesini değerlendirir |
| Çok sınıflı ve dengesiz mi? | Macro ve weighted F1 birlikte | Nadir sınıfların gizlenmesini önler |
| Operasyonel karar mı verilecek? | Maliyet matrisi + eşik analizi | Metrik iş etkisine bağlanır |


# Bölüm B — Regresyon Metrikleri

Regresyonda gerçek değer $y_i$, tahmin $\hat{y}_i$ ve residual/hata:

$$e_i=y_i-\hat{y}_i$$

İşaret, modelin fazla mı az mı tahmin ettiğini gösterir. Hatanın büyüklüğünü tek sayıya indirgerken mutlak değer, kare, oran veya logaritma kullanabilirim. Her dönüşüm farklı hataları öne çıkarır.


In [ ]:
# İki örnek regresyon projesi
claim_true = np.array([4_000, 7_500, 12_000, 18_000, 24_000, 40_000, 65_000, 110_000], dtype=float)
claim_pred = np.array([5_200, 6_800, 10_500, 20_500, 20_000, 44_000, 58_000, 92_000], dtype=float)

demand_true = np.array([520, 610, 590, 680, 720, 760, 810, 790, 860, 900], dtype=float)
demand_pred = np.array([500, 640, 570, 650, 750, 730, 850, 770, 830, 940], dtype=float)

residual_df = pd.DataFrame({
    "gercek": claim_true,
    "tahmin": claim_pred,
    "residual": claim_true - claim_pred,
    "mutlak_hata": np.abs(claim_true - claim_pred),
})
residual_df


## 16. MAE — ortalama mutlak hata

$$MAE=\frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|$$

### Adım adım

1. Her gözlemde gerçek değer ile tahmin farkını alırım.
2. Artı ve eksi hataların birbirini götürmemesi için mutlak değer kullanırım.
3. Mutlak hataları ortalarım.

MAE hedef değişkenle aynı birimdedir ve doğrudan yorumlanabilir. Büyük hataları MSE/RMSE kadar sert cezalandırmaz.

### İki proje çıktısını yorumlama

**Proje 1 — Hasar tutarı:** MAE=1.250 TL ise poliçe başına tahminim gerçek hasar tutarından ortalama 1.250 TL sapmaktadır. Dağılım çarpıksa medyan ve yüksek yüzdelik hatalara da bakarım.

**Proje 2 — Günlük satış tahmini:** MAE=420 ürün ise günlük tahminler ortalama 420 adet sapmaktadır. Ortalama satış 20.000 ise bağlamsal hata yaklaşık %2,1 düzeyindedir.


In [ ]:
def mae(y, yhat):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(yhat))))

print(f"Hasar MAE: {mae(claim_true, claim_pred):,.2f} TL")
print(f"Talep MAE: {mae(demand_true, demand_pred):,.2f} adet")


## 17. MSE — büyük hataları karesel olarak büyütmek

$$MSE=\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2$$

Kare alma nedeniyle büyük hatalar metriği hızla büyütür. Optimizasyon açısından türevlenebilir ve kullanışlıdır; fakat birimi hedefin karesidir. TL tahmininde sonuç $TL^2$ olur ve doğrudan iş yorumu zordur.

### İki proje çıktısını yorumlama

**Proje 1 — Enerji talebi:** Model A MSE=6.400 $kWh^2$, Model B MSE=2.500 $kWh^2$ ise B büyük hataları daha iyi kontrol etmektedir. Kare birim nedeniyle iş yorumunu RMSE ile tamamlarım.

**Proje 2 — Hasar tutarı:** MSE=4.800.000 $TL^2$ tek başına “ortalama 4,8 milyon TL hata” anlamına gelmez. Birkaç uç hasar bu değeri büyütmüş olabilir; residual dağılımını ve RMSE'yi incelerim.


In [ ]:
def mse(y, yhat):
    return float(np.mean((np.asarray(y) - np.asarray(yhat)) ** 2))

print(f"Hasar MSE: {mse(claim_true, claim_pred):,.2f} TL²")
print(f"Talep MSE: {mse(demand_true, demand_pred):,.2f} adet²")


## 18. RMSE — karesel cezayı hedefin birimine geri getirmek

$$RMSE=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}$$

RMSE, MSE'nin kareköküdür. Hedefle aynı birime döner; büyük hatalara MAE'den daha duyarlıdır.

### İki proje çıktısını yorumlama

**Proje 1 — Mağaza talebi:** RMSE=80 ürün ve ortalama talep 800 ise tipik karesel hata ölçeği ortalamanın yaklaşık %10'udur. “Tahminlerin %10'u yanlış” demek değildir.

**Proje 2 — Araç onarım maliyeti:** RMSE=3.200 TL, MAE=1.400 TL ise RMSE'nin belirgin biçimde büyük olması bazı tahminlerde çok yüksek hata bulunduğuna işaret eder.


In [ ]:
def rmse(y, yhat):
    return float(np.sqrt(mse(y, yhat)))

print(f"Hasar RMSE: {rmse(claim_true, claim_pred):,.2f} TL")
print(f"Talep RMSE: {rmse(demand_true, demand_pred):,.2f} adet")


## 19. $R^2$ — ortalamaya göre ne kadar açıklama gücü kazandım?

$$R^2=1-\frac{\sum_i(y_i-\hat{y}_i)^2}{\sum_i(y_i-\bar{y})^2}$$

- Pay: modelin karesel hata toplamı
- Payda: herkese hedef ortalamasını söyleyen basit modelin karesel hata toplamı

$R^2=1$ kusursuz tahmindir. $R^2=0$ ortalama tahmini kadar performans demektir. Test setinde negatif $R^2$, modelin ortalama söylemekten daha kötü olduğunu gösterir.

### İki proje çıktısını yorumlama

**Proje 1 — Konut fiyatı:** Test $R^2=0,84$ ise test setindeki fiyat değişkenliğinin %84'ü model tahminleriyle açıklanabilmektedir. Bu sonuç nedensellik anlamına gelmez ve mutlak hata büyüklüğünü göstermez.

**Proje 2 — Hasar şiddeti:** Train $R^2=0,72$, test $R^2=-0,12$ ise model eğitim verisine aşırı uyum sağlamış olabilir. Testte ortalama hasar tutarını tahmin etmek modelden daha iyidir.


In [ ]:
def r2(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    ss_res = np.sum((y - yhat) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    return float(1 - ss_res / ss_tot) if ss_tot else np.nan

print(f"Hasar R²: {r2(claim_true, claim_pred):.3f}")
print(f"Talep R²: {r2(demand_true, demand_pred):.3f}")


## 20. MAPE — yüzde hata, fakat sıfır problemi var

$$MAPE=\frac{100}{n}\sum_{i=1}^{n}\left|\frac{y_i-\hat{y}_i}{y_i}\right|$$

MAPE ölçekten bağımsız ve yüzde olarak kolay anlatılır. Fakat:

- $y_i=0$ ise tanımsızdır,
- $y_i$ sıfıra yakınsa aşırı büyür,
- fazla ve eksik tahminleri simetrik cezalandırmaz.

### İki proje çıktısını yorumlama

**Proje 1 — Aylık talep:** MAPE=%8,4 ise tahminler gerçek değerden ortalama mutlak yüzde olarak %8,4 sapmaktadır. Seride sıfır ve çok küçük değer yoksa yorum kullanışlıdır.

**Proje 2 — Yeni ürün satışı:** Gerçek satış 1, tahmin 4 ise tek gözlemin yüzde hatası %300'dür. MAPE=%145 gibi bir sonuç modelin tamamından çok küçük paydaların etkisini yansıtabilir.


In [ ]:
def mape(y, yhat):
    y = np.asarray(y, dtype=float)
    yhat = np.asarray(yhat, dtype=float)
    if np.any(y == 0):
        return np.nan
    return float(100 * np.mean(np.abs((y - yhat) / y)))

print(f"Hasar MAPE: %{mape(claim_true, claim_pred):.2f}")
print(f"Talep MAPE: %{mape(demand_true, demand_pred):.2f}")
print("Sıfır içeren seri:", mape([0, 10, 20], [2, 9, 18]))


## 21. sMAPE — paydayı gerçek ve tahminle simetrikleştirmek

Yaygın tanımlardan biri:

$$sMAPE=\frac{100}{n}\sum_{i=1}^{n}
\frac{2|y_i-\hat{y}_i|}{|y_i|+|\hat{y}_i|}$$

Bu tanımda sonuç genel olarak %0 ile %200 arasındadır. Hem gerçek hem tahmin 0 ise gözlem için özel karar gerekir. sMAPE, MAPE'nin bütün sorunlarını tamamen çözmez; özellikle düşük hacimlerde hâlâ oynaktır.

### İki proje çıktısını yorumlama

**Proje 1 — Çağrı hacmi:** sMAPE=%9,8 ise gerçek ve tahmin ölçeğine göre simetrik ortalama hata yaklaşık %9,8'dir. Dönemler arası model karşılaştırmasında kullanılabilir.

**Proje 2 — Düşük hacimli şube:** sMAPE=%38 ise göreli hata yüksektir. Şubede günlük hacimler 0–5 arasındaysa birkaç çağrılık fark skoru büyütebilir; MAE ile birlikte raporlarım.


In [ ]:
def smape(y, yhat):
    y = np.asarray(y, dtype=float)
    yhat = np.asarray(yhat, dtype=float)
    denominator = np.abs(y) + np.abs(yhat)
    ratio = np.divide(2 * np.abs(y - yhat), denominator,
                      out=np.zeros_like(denominator), where=denominator != 0)
    return float(100 * np.mean(ratio))

print(f"Hasar sMAPE: %{smape(claim_true, claim_pred):.2f}")
print(f"Talep sMAPE: %{smape(demand_true, demand_pred):.2f}")


## 22. RMSLE — mutlak farktan çok oransal fark önemliyse

$$RMSLE=\sqrt{\frac{1}{n}\sum_{i=1}^{n}
\left[\log(1+\hat{y}_i)-\log(1+y_i)\right]^2}$$

RMSLE büyük hedef değerlerinin etkisini logaritmayla sıkıştırır ve çarpımsal/oransal hatalara odaklanır. Negatif gerçek veya tahmin değerlerinde kullanılamaz. Sistematik düşük tahminleri bazı durumlarda farklı biçimde cezalandırabilir.

Yaklaşık çarpan yorumu için $e^{RMSLE}-1$ kullanılabilir; bu kesin bir yüzde hata eşitliği değildir.

### İki proje çıktısını yorumlama

**Proje 1 — Müşteri yaşam boyu değeri:** RMSLE=0,28 ise tipik çarpımsal sapma kabaca $e^{0,28}-1\approx\%32$ ölçeğindedir. Düşük ve yüksek değerleri aynı mutlak TL ağırlığıyla değerlendirmemiş olurum.

**Proje 2 — Günlük sipariş sayısı:** RMSLE=0,55 yaklaşık %73'lük çarpımsal sapma ölçeğine karşılık gelir. Siparişler negatif olmadığı için uygundur; sıfır değerler $\log(1+y)$ sayesinde işlenebilir.


In [ ]:
def rmsle(y, yhat):
    y = np.asarray(y, dtype=float)
    yhat = np.asarray(yhat, dtype=float)
    if np.any(y < 0) or np.any(yhat < 0):
        return np.nan
    return float(np.sqrt(np.mean((np.log1p(yhat) - np.log1p(y)) ** 2)))

claim_rmsle = rmsle(claim_true, claim_pred)
print(f"Hasar RMSLE: {claim_rmsle:.3f}")
print(f"Yaklaşık çarpımsal sapma ölçeği: %{(np.exp(claim_rmsle) - 1) * 100:.1f}")


## 23. WAPE — toplam hacme göre ağırlıklı mutlak hata

$$WAPE=100\times\frac{\sum_i|y_i-\hat{y}_i|}{\sum_i|y_i|}$$

WAPE, toplam mutlak hatayı toplam gerçek hacme böler. Hacimli gözlemler metriği daha fazla etkiler. Toplam gerçek hacim 0 ise tanımsızdır. Büyük şubelerin küçük şubelerdeki kötü performansı maskeleyebileceğini unutmam.

### İki proje çıktısını yorumlama

**Proje 1 — Ulusal satış tahmini:** WAPE=%6,2 ise toplam mutlak hata, toplam gerçekleşen satış hacminin %6,2'sidir. Bütçe ve kapasite planlamasında anlaşılır bir özet sağlar.

**Proje 2 — Şube tahmini:** Genel WAPE=%5 iken küçük şubelerde MAPE=%40 olabilir. Büyük şubeler toplam hacmi domine ettiği için iyi genel WAPE, her şubenin iyi tahmin edildiğini göstermez.


In [ ]:
def wape(y, yhat):
    y = np.asarray(y, dtype=float)
    yhat = np.asarray(yhat, dtype=float)
    denominator = np.sum(np.abs(y))
    return float(100 * np.sum(np.abs(y - yhat)) / denominator) if denominator else np.nan

print(f"Hasar WAPE: %{wape(claim_true, claim_pred):.2f}")
print(f"Talep WAPE: %{wape(demand_true, demand_pred):.2f}")


## Regresyon hatalarını görselleştirme

Tek bir skor bütün hata yapısını göstermez. Bu nedenle en azından şu iki grafiğe bakarım:

1. Gerçek değer–tahmin grafiği
2. Tahmin–residual grafiği

Residual'ların sıfır çevresinde rastgele dağılmasını beklerim. Huni şekli değişen varyansa, eğri desen eksik doğrusal olmayan yapıya, tekil büyük noktalar uç hatalara işaret edebilir.


In [ ]:
residuals = claim_true - claim_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

axes[0].scatter(claim_true, claim_pred, s=70, alpha=0.8)
limit = max(claim_true.max(), claim_pred.max())
axes[0].plot([0, limit], [0, limit], "--", color="gray")
axes[0].set_title("Gerçek değer ve tahmin")
axes[0].set_xlabel("Gerçek hasar tutarı")
axes[0].set_ylabel("Tahmin")

axes[1].scatter(claim_pred, residuals, s=70, alpha=0.8, color="darkorange")
axes[1].axhline(0, linestyle="--", color="gray")
axes[1].set_title("Residual grafiği")
axes[1].set_xlabel("Tahmin")
axes[1].set_ylabel("Gerçek - tahmin")

plt.tight_layout()
plt.show()


## Bütün regresyon metriklerini birlikte hesaplayalım

Tek bir metriği “kazanan” ilan etmek yerine, iş problemine uygun küçük bir metrik seti raporlamak daha güvenlidir.


In [ ]:
def regression_report(y, yhat):
    return pd.Series({
        "MAE": mae(y, yhat),
        "MSE": mse(y, yhat),
        "RMSE": rmse(y, yhat),
        "R2": r2(y, yhat),
        "MAPE_%": mape(y, yhat),
        "sMAPE_%": smape(y, yhat),
        "RMSLE": rmsle(y, yhat),
        "WAPE_%": wape(y, yhat),
    })

report = pd.DataFrame({
    "Hasar tutarı projesi": regression_report(claim_true, claim_pred),
    "Talep tahmini projesi": regression_report(demand_true, demand_pred),
})
report


## Regresyon için kısa seçim rehberi

| İhtiyaç | Öncelikli metrik | Dikkat |
|---|---|---|
| Ortalama hatayı hedef biriminde anlatmak | MAE | Büyük hataları daha yumuşak cezalandırır |
| Büyük hataları özellikle cezalandırmak | RMSE | Uç değerlere duyarlıdır |
| Optimizasyonda karesel kayıp kullanmak | MSE | Birimi hedefin karesidir |
| Ortalama modele göre açıklama gücü | $R^2$ | Mutlak hata büyüklüğünü göstermez |
| Yüzde hata ve sıfır olmayan seri | MAPE | Sıfır/küçük gerçeklerde bozulur |
| Gerçek ve tahmini simetrik oranlamak | sMAPE | Düşük hacimde oynaktır |
| Çarpımsal hata ve pozitif hedef | RMSLE | Negatif değerlerde kullanılamaz |
| Toplam hacme göre hata | WAPE | Büyük gruplar küçük grupları maskeleyebilir |


# Bölüm C — Metrik seçerken yaptığım yaygın hatalar

1. **Accuracy'yi tek başına raporlamak:** Sınıf dengesizliğini gizleyebilir.
2. **ROC-AUC'yi operasyonel başarı sanmak:** AUC sıralamayı ölçer; seçilen eşikteki precision/recall'u söylemez.
3. **PR-AUC'yi pozitif oranından bağımsız yorumlamak:** Referans seviye prevalence'tır.
4. **F1'i iş maliyeti yerine kullanmak:** F1, FP ve FN'ye eşit ağırlık verir; gerçek maliyetler eşit olmayabilir.
5. **$R^2$ yüksekse hatayı küçük sanmak:** Hedef ölçeği büyükse yüksek $R^2$ ile yüksek TL hatası birlikte görülebilir.
6. **MAPE'yi sıfırlı seride kullanmak:** Metrik tanımsız veya aşırı oynak hale gelir.
7. **Train metriğini başarı kabul etmek:** Model seçimi validation, son rapor bağımsız test veya zaman bazlı holdout üzerinde yapılmalıdır.
8. **Alt grupları görmezden gelmek:** Genel metrik iyi iken bölge, ürün veya müşteri grubunda performans kötü olabilir.


# Mini vaka — doğru metriği seçelim

Bir motor sigortası projesinde:

- hasar oranı %4,
- yanlış alarmın inceleme maliyeti 30 TL,
- kaçırılan hasarın beklenen maliyeti 800 TL,
- model bir hasar olasılığı üretiyor.

### Tartışma soruları

1. Accuracy neden yeterli değildir?
2. Recall mu precision mı daha kritik görünmektedir?
3. ROC-AUC ve PR-AUC bize hangi farklı bilgileri verir?
4. Eşik seçiminde hangi maliyet fonksiyonunu kurarım?
5. Olasılıkların fiyatlama veya rezerv için kullanılacağı durumda hangi metrikleri eklerim?

### Benim çözüm yaklaşımım

- Model sıralamasını ROC-AUC ve özellikle PR-AUC ile kontrol ederim.
- Eşik bazında precision, recall, confusion matrix ve beklenen maliyeti raporlarım.
- $30\times FP+800\times FN$ maliyetini eşiklere göre hesaplarım.
- Olasılık kullanılacaksa log-loss, Brier score ve kalibrasyon grafiğini eklerim.
- Son kararı yalnızca tek metrikle değil, iş kapasitesi ve maliyetle birlikte veririm.


In [ ]:
# Mini vaka için örnek eşik maliyeti simülasyonu
rng = np.random.default_rng(RANDOM_STATE)
n = 5_000
claim_y = rng.binomial(1, 0.04, size=n)

# Gerçek hasarlarda biraz daha yüksek skor üreten temsili bir model
claim_score = np.where(
    claim_y == 1,
    rng.beta(3.2, 4.0, size=n),
    rng.beta(1.2, 9.0, size=n),
)

cost_table = pd.DataFrame([
    metrics_at_threshold(claim_y, claim_score, t, fp_cost=30, fn_cost=800)
    for t in np.arange(0.05, 0.81, 0.025)
])

best_row = cost_table.loc[cost_table["business_cost"].idxmin()]
display(cost_table.sort_values("business_cost").head(8).round(3))
print(f"Bu örnek maliyetlerle en düşük maliyetli eşik: {best_row['threshold']:.3f}")

plt.figure(figsize=(7, 4.5))
plt.plot(cost_table["threshold"], cost_table["business_cost"], marker="o", markersize=3)
plt.axvline(best_row["threshold"], linestyle="--", color="firebrick", label="Minimum maliyet")
plt.xlabel("Karar eşiği")
plt.ylabel("Toplam örnek maliyet")
plt.title("Eşiği iş maliyetiyle seçmek")
plt.legend()
plt.tight_layout()
plt.show()


# Vize öncesi hızlı tekrar

## Sınıflandırma

- Accuracy: bütün doğruların oranı
- Precision: pozitif dediklerimin isabeti
- Recall: gerçek pozitifleri yakalama oranı
- Specificity: gerçek negatifleri doğru eleme oranı
- F1: precision–recall dengesi
- Balanced accuracy: recall ve specificity ortalaması
- MCC: confusion matrix'in dört hücresini birlikte kullanan skor
- ROC-AUC: pozitifleri negatiflerin önüne sıralama
- PR-AUC: nadir pozitiflerde precision–recall ilişkisi
- Log-loss ve Brier: olasılık tahminlerinin kalitesi

## Regresyon

- MAE: doğrudan ve dayanıklı ortalama hata
- MSE/RMSE: büyük hatalara daha duyarlı
- $R^2$: ortalama modele göre açıklama gücü
- MAPE/sMAPE: göreli hata; düşük ve sıfır değerlerde dikkat
- RMSLE: çarpımsal hata
- WAPE: toplam hacme göre hata

## Ana fikir

> En iyi metrik yoktur. İş problemindeki en pahalı hatayı görünür kılan metrik vardır.


## Ders sonu kısa kontrol soruları

1. Pozitif sınıf oranı %2 iken accuracy neden yanıltıcı olabilir?
2. Precision yüksek, recall düşükse model nasıl davranıyordur?
3. Specificity hangi hata türünü görünür kılar?
4. ROC-AUC=0,80 ifadesi neden “%80 doğru tahmin” değildir?
5. PR-AUC yorumunda neden pozitif sınıf oranını bilmek gerekir?
6. Log-loss yanlış ve aşırı güvenli tahmini neden daha ağır cezalandırır?
7. RMSE'nin MAE'den çok büyük olması ne düşündürür?
8. Negatif test $R^2$ ne anlama gelir?
9. MAPE hangi durumda kullanılamaz veya yanıltıcı olur?
10. Karar eşiğini neden varsayılan 0,50'ye bırakmamalıyım?
